# Predictive distributions of outstanding liabilities (England & Verrall 2006)

Adapted from Peter England's *EV_2006_PredictiveDistributions* notebook in
the [StochasticReserving repository](https://github.com/DrPeterEngland/StochasticReserving)
(`Python_Examples/EV_2006_PredictiveDistributions.ipynb`, MIT licence,
provided by EMC Actuarial and Analytics Ltd), which reproduces England &
Verrall (2006), *Predictive distributions of outstanding liabilities in
general insurance*, Annals of Actuarial Science 1(II)
([https://doi.org/10.1017/S1748499500000142](https://doi.org/10.1017/S1748499500000142)). The code uses `bayesianchainladder`; the commentary is
paraphrased from the paper and notebook.

England's ODP model has a *constant* scale parameter (unlike EVW 2019's
non-constant scale) — this is the classic over-dispersed Poisson bootstrap
with a single dispersion for the whole triangle. The triangle is the same
Taylor & Ashe (1983) incremental paid losses used throughout this package's
England-derived notebooks.

As elsewhere, gaps below the Monte Carlo / MCMC tables are ordinary
simulation noise (same algorithm, different RNG stream and seed sequence);
MCMC parameter comparisons additionally reflect England's flat priors versus
this package's weakly informative `Normal(0, 10)` priors, and the
negative-binomial reserve comparison reflects a different variance function
(`Var = mu + mu^2/k` versus ODP's `Var = phi * mu`).

## 1. Setup

In [1]:
import json
import warnings
from importlib import resources

import numpy as np
import pandas as pd
import pymc as pm

from bayesianchainladder import (
    BayesianChainLadderGLM,
    BayesianMackChainLadder,
    CorrelatedBootstrapChainLadder,
    build_quasi_poisson_model,
    load_england_sample,
    odp_analytic_rmsep,
    prepare_model_data,
)
from bayesianchainladder._triangle_ops import cumulative_array, cumulative_to_incremental
from bayesianchainladder.analytic import _design_matrix

warnings.filterwarnings("ignore", category=FutureWarning)
pd.options.display.float_format = "{:,.0f}".format

tri = load_england_sample("taylor_ashe")
origins = list(tri.origin.year)

with resources.files("bayesianchainladder.data").joinpath(
    "england_reference_values.json"
).open("r", encoding="utf-8") as fh:
    ref_all = json.load(fh)
ref = ref_all["ev_2006"]
ref_evw = ref_all["evw_2019"]

SEED = 101
N_SIMS = 10_000
MCMC_KWARGS = dict(draws=1000, tune=1000, chains=2, random_seed=SEED)


def compare(ours, england, labels=None):
    """Side-by-side comparison table: ours, England, diff %."""
    ours = np.asarray(ours, dtype=float)
    england = np.asarray(england, dtype=float)
    if labels is None:
        labels = list(range(len(ours)))
    with np.errstate(divide="ignore", invalid="ignore"):
        diff_pct = np.where(england != 0, 100.0 * (ours - england) / england, np.nan)
    return pd.DataFrame(
        {"ours": ours, "England": england, "diff %": diff_pct}, index=labels
    )


COMPARE_FMT = {"ours": "{:,.0f}", "England": "{:,.0f}", "diff %": "{:+.1f}%"}
DECIMAL_FMT = {"ours": "{:,.3f}", "England": "{:,.3f}", "diff %": "{:+.1f}%"}

## 2. Maximum likelihood: ODP with constant scale

In [2]:
odp = odp_analytic_rmsep(tri, scale="constant")
latest = tri.latest_diagonal.values[0, 0, :, 0]
ultimate = latest + odp.reserves

mle_table = pd.DataFrame(
    {"latest": latest, "reserve": odp.reserves, "ultimate": ultimate, "sd": odp.reserve_sd},
    index=origins,
)
mle_table["cov"] = np.where(mle_table["reserve"] != 0, mle_table["sd"] / mle_table["reserve"], np.nan)
display(mle_table.style.format({"latest": "{:,.0f}", "reserve": "{:,.0f}", "ultimate": "{:,.0f}",
                                 "sd": "{:,.0f}", "cov": "{:.1%}"}))

t_ml = ref["ml_analytic_odp_constant"]
labels = origins + ["Total"]
display(compare(list(odp.reserves) + [odp.total_reserve], t_ml["reserves"] + [t_ml["total_reserve"]], labels).style.format(COMPARE_FMT))
display(compare(list(odp.reserve_sd) + [odp.total_sd], t_ml["reserve_sd"] + [t_ml["total_sd"]], labels).style.format(COMPARE_FMT))

params = ref["ml_parameters"]
display(compare(odp.coefficients, params["estimate"], params["labels"]).style.format(DECIMAL_FMT))
print(f"sqrt(scale) = {np.sqrt(odp.scale[0]):.2f} (England: {t_ml['sqrt_scale']})")

,latest,reserve,ultimate,sd,cov
2001,"3,901,463",0,"3,901,463",0,nan%
2002,"5,339,085","94,634","5,433,719","110,099",116.3%
2003,"4,909,315","469,511","5,378,826","216,042",46.0%
2004,"4,588,268","709,638","5,297,906","260,871",36.8%
2005,"3,873,311","984,889","4,858,200","303,549",30.8%
2006,"3,691,712","1,419,459","5,111,171","375,012",26.4%
2007,"3,483,130","2,177,641","5,660,771","495,376",22.7%
2008,"2,864,498","3,920,301","6,784,799","789,957",20.2%
2009,"1,363,294","4,278,972","5,642,266","1,046,508",24.5%
2010,"344,014","4,625,811","4,969,825","1,980,091",42.8%


,ours,England,diff %
2001,0,0,+nan%
2002,"94,634","94,634",-0.0%
2003,"469,511","469,511",+0.0%
2004,"709,638","709,638",-0.0%
2005,"984,889","984,889",-0.0%
2006,"1,419,459","1,419,459",+0.0%
2007,"2,177,641","2,177,641",-0.0%
2008,"3,920,301","3,920,301",+0.0%
2009,"4,278,972","4,278,972",+0.0%
2010,"4,625,811","4,625,811",-0.0%


,ours,England,diff %
2001,0,0,+nan%
2002,"110,099","110,099",+0.0%
2003,"216,042","216,042",+0.0%
2004,"260,871","260,871",-0.0%
2005,"303,549","303,549",-0.0%
2006,"375,012","375,012",+0.0%
2007,"495,376","495,376",-0.0%
2008,"789,957","789,957",+0.0%
2009,"1,046,508","1,046,508",+0.0%
2010,"1,980,091","1,980,091",-0.0%


,ours,England,diff %
Intercept,12.506,12.506,+0.0%
OP2,0.331,0.331,+0.1%
OP3,0.321,0.321,+0.0%
OP4,0.306,0.306,-0.0%
OP5,0.219,0.219,+0.1%
OP6,0.270,0.270,+0.0%
OP7,0.372,0.372,+0.1%
OP8,0.553,0.553,+0.1%
OP9,0.369,0.369,-0.0%
OP10,0.242,0.242,+0.0%


sqrt(scale) = 229.35 (England: 229.35)


**Commentary.** The ODP GLM is fit by IRLS in both cases, so the reserves,
standard errors and coefficient estimates match England's to the precision
shown — this is a maximum-likelihood fit, not a simulation, so there is no
Monte Carlo noise to explain any residual gap.

## 3. Standard errors of the ML parameters

In [3]:
# `odp_analytic_rmsep` does not expose parameter standard errors on
# `AnalyticResult`; these are the same ingredients its IRLS fit uses
# internally (design matrix, fitted mean, scale), assembled here directly.
cum, _, _ = cumulative_array(tri)
incr = cumulative_to_incremental(cum)
n_o, n_d = incr.shape
X, i_all, j_all = _design_matrix(n_o, n_d)
obs_mask = ~np.isnan(incr).ravel()
mu_all = np.exp(X @ odp.coefficients)
mu_obs = mu_all[obs_mask]
phi_obs = odp.scale[j_all[obs_mask]]
X_obs = X[obs_mask]
parameter_se = np.sqrt(np.diag(np.linalg.inv((X_obs.T * (mu_obs / phi_obs)) @ X_obs)))

display(compare(parameter_se, params["standard_error"], params["labels"]).style.format(DECIMAL_FMT))

,ours,England,diff %
Intercept,0.173,0.173,-0.0%
OP2,0.154,0.154,-0.3%
OP3,0.158,0.158,-0.2%
OP4,0.161,0.161,-0.2%
OP5,0.168,0.168,-0.0%
OP6,0.171,0.171,-0.1%
OP7,0.174,0.174,+0.3%
OP8,0.187,0.187,-0.3%
OP9,0.239,0.239,+0.1%
OP10,0.428,0.428,-0.1%


**Commentary.** These standard errors mirror what `odp_analytic_rmsep` does
internally to build its reserve covariance matrix, evaluated here at the
parameter level instead. They match England's GLM standard errors closely.

## 4. ODP bootstrap

In [4]:
boot = CorrelatedBootstrapChainLadder(n_sims=N_SIMS, rho=0.0, random_seed=SEED).fit(tri)
summary = boot.summary_statistics("reserves")

t_boot = ref["bootstrap_odp_constant"]
display(compare(summary["mean"], t_boot["avg_reserves"] + [t_boot["total_avg_reserve"]], labels).style.format(COMPARE_FMT))
display(compare(summary["std"], t_boot["sd"] + [t_boot["total_sd"]], labels).style.format(COMPARE_FMT))
display(compare(summary["cov"] * 100, t_boot["cov_pct"] + [t_boot["total_cov_pct"]], labels).style.format(
    {"ours": "{:.1f}", "England": "{:.1f}", "diff %": "{:+.1f}%"}
))

total_row = summary.loc["Total"]
ours_dict = {
    "min": total_row["min"], "p0.5": total_row["0.5%"], "p1": total_row["1%"], "p5": total_row["5%"],
    "p10": total_row["10%"], "p25": total_row["25%"], "p50": total_row["50%"], "p75": total_row["75%"],
    "p90": total_row["90%"], "p95": total_row["95%"], "p99": total_row["99%"], "p99.5": total_row["99.5%"],
    "max": total_row["max"],
}
ref_total = t_boot["total_summary"]
display(compare(list(ours_dict.values()), [ref_total[k] for k in ours_dict], list(ours_dict.keys())).style.format(COMPARE_FMT))

/Users/atroyer/Projects/bayesianchainladder/.claude/worktrees/stochastic-reserving-review-61d253/.venv/lib/python3.12/site-packages/chainladder/utils/weighted_regression.py:84: RuntimeWarning: invalid value encountered in sqrt
  residual = (y - fitted_value) * xp.sqrt(w)
/Users/atroyer/Projects/bayesianchainladder/.claude/worktrees/stochastic-reserving-review-61d253/.venv/lib/python3.12/site-packages/chainladder/development/development.py:174: RuntimeWarning: invalid value encountered in sqrt
  / xp.swapaxes(xp.sqrt(x ** (2 - exponent))[..., 0:1, :], -1, -2)
/Users/atroyer/Projects/bayesianchainladder/.claude/worktrees/stochastic-reserving-review-61d253/.venv/lib/python3.12/site-packages/chainladder/development/development.py:185: RuntimeWarning: invalid value encountered in sqrt
  std = xp.sqrt((1 / num_to_nan(w)) * (self.sigma_**2).values)


,ours,England,diff %
2001,0,0,+nan%
2002,"94,861","93,572",+1.4%
2003,"469,165","473,108",-0.8%
2004,"709,668","712,923",-0.5%
2005,"984,409","989,285",-0.5%
2006,"1,423,506","1,433,274",-0.7%
2007,"2,186,153","2,183,609",+0.1%
2008,"3,928,632","3,937,888",-0.2%
2009,"4,317,065","4,314,944",+0.0%
2010,"4,746,968","4,727,535",+0.4%


,ours,England,diff %
2001,0,0,+nan%
2002,"115,542","109,301",+5.7%
2003,"218,650","220,092",-0.7%
2004,"259,261","260,734",-0.6%
2005,"306,100","305,872",+0.1%
2006,"380,813","377,821",+0.8%
2007,"497,813","492,788",+1.0%
2008,"795,714","786,556",+1.2%
2009,"1,069,418","1,049,731",+1.9%
2010,"2,037,322","2,007,248",+1.5%


,ours,England,diff %
2001,nan,0.0,+nan%
2002,121.8,116.8,+4.3%
2003,46.6,46.5,+0.2%
2004,36.5,36.6,-0.2%
2005,31.1,30.9,+0.6%
2006,26.8,26.4,+1.3%
2007,22.8,22.6,+0.8%
2008,20.3,20.0,+1.3%
2009,24.8,24.3,+1.9%
2010,42.9,42.5,+1.0%


,ours,England,diff %
min,"9,775,686","10,663,383",-8.3%
p0.5,"11,953,675","12,635,166",-5.4%
p1,"12,371,707","13,071,145",-5.4%
p5,"14,113,398","14,416,307",-2.1%
p10,"15,113,048","15,225,145",-0.7%
p25,"16,820,668","16,774,553",+0.3%
p50,"18,735,981","18,619,965",+0.6%
p75,"20,743,063","20,681,900",+0.3%
p90,"22,770,981","22,761,694",+0.0%
p95,"24,028,694","24,114,519",-0.4%


**Commentary.** England draws Gamma pseudo-data for the bootstrap resampling
stage; this package draws Normal pseudo-data (falling back to Normal only
where the ODP mean is non-positive) with the same Gamma process-error stage.
Combined with ordinary Monte Carlo error at 10,000 simulations, this
explains the residual few-percent gap throughout this table.

## 5. MCMC: England's quasi-Poisson model

In [5]:
obs, _ = prepare_model_data(tri)
qp_model = build_quasi_poisson_model(obs, scale=float(odp.scale[0]))
with qp_model:
    qp_idata = pm.sample(**MCMC_KWARGS, progressbar=False)

intercept_mean = float(qp_idata.posterior["intercept"].mean())
intercept_sd = float(qp_idata.posterior["intercept"].std())
alpha_mean = qp_idata.posterior["alpha_raw"].mean(("chain", "draw")).values
alpha_sd = qp_idata.posterior["alpha_raw"].std(("chain", "draw")).values
beta_mean = qp_idata.posterior["beta_raw"].mean(("chain", "draw")).values
beta_sd = qp_idata.posterior["beta_raw"].std(("chain", "draw")).values

posterior_mean = np.concatenate([[intercept_mean], alpha_mean, beta_mean])
posterior_sd = np.concatenate([[intercept_sd], alpha_sd, beta_sd])

mcmc_params = ref["mcmc_parameters"]
qp_table = pd.DataFrame(
    {"ML": params["estimate"], "England MCMC": mcmc_params["posterior_mean"], "ours MCMC": posterior_mean},
    index=params["labels"],
)
display(qp_table.style.format("{:.3f}"))
display(compare(posterior_mean, mcmc_params["posterior_mean"], params["labels"]).style.format(DECIMAL_FMT))
display(compare(posterior_sd, mcmc_params["posterior_sd"], params["labels"]).style.format(DECIMAL_FMT))

Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (2 chains in 2 jobs)


NUTS: [intercept, alpha_raw, beta_raw]


Sampling 2 chains for 1_000 tune and 1_000 draw iterations (2_000 + 2_000 draws total) took 1 seconds.


We recommend running at least 4 chains for robust computation of convergence diagnostics


,ML,England MCMC,ours MCMC
Intercept,12.506,12.489,12.491
OP2,0.331,0.332,0.335
OP3,0.321,0.323,0.329
OP4,0.306,0.308,0.314
OP5,0.219,0.220,0.224
OP6,0.270,0.270,0.276
OP7,0.372,0.371,0.381
OP8,0.553,0.552,0.561
OP9,0.369,0.362,0.366
OP10,0.242,0.183,0.165


,ours,England,diff %
Intercept,12.491,12.489,+0.0%
OP2,0.335,0.332,+0.9%
OP3,0.329,0.323,+1.8%
OP4,0.314,0.308,+2.0%
OP5,0.224,0.220,+1.8%
OP6,0.276,0.270,+2.3%
OP7,0.381,0.371,+2.7%
OP8,0.561,0.552,+1.6%
OP9,0.366,0.362,+1.1%
OP10,0.165,0.183,-10.0%


,ours,England,diff %
Intercept,0.173,0.174,-0.4%
OP2,0.159,0.154,+3.5%
OP3,0.162,0.156,+4.0%
OP4,0.165,0.161,+2.8%
OP5,0.176,0.168,+4.8%
OP6,0.175,0.172,+1.5%
OP7,0.179,0.176,+1.9%
OP8,0.189,0.187,+1.1%
OP9,0.239,0.242,-1.3%
OP10,0.437,0.441,-0.9%


**Commentary.** `build_quasi_poisson_model` implements the same
over-dispersed-Poisson quasi-likelihood as England's Stan model, with
`Normal(0, 10)` priors in place of his flat priors. With this much data
relative to a wide prior, the posterior means and SDs are close to both the
ML estimate and England's own MCMC. Most coefficients agree to within about
2%; DP10 (the most poorly identified coefficient — only one observation
contributes to it) differs by about 10% in absolute terms of a small
coefficient, and DP6, DP7 and DP9 show large *percentage* differences (up to
about 48%) only because those posterior means are themselves close to zero
— the absolute gaps there (0.006-0.009) are no larger than elsewhere.

## 6. MCMC reserves: the closest bayesianchainladder estimator

In [6]:
glm = BayesianChainLadderGLM(
    formula="incremental ~ 1 + C(origin) + C(dev)",
    family="negativebinomial",
    **MCMC_KWARGS,
).fit(tri)
glm_summary = glm.summary_statistics("reserves")
# Origin 2001 is fully developed (no future cells), so BayesianChainLadderGLM
# never predicts it and it is absent from reserves_posterior_; drop it from
# the England side of the comparison too (its reserve is 0 there as well).
labels_glm = origins[1:] + ["Total"]

t_mcmc = ref["mcmc_odp_constant"]
display(compare(glm_summary["mean"], t_mcmc["avg_reserves"][1:] + [t_mcmc["total_avg_reserve"]], labels_glm).style.format(COMPARE_FMT))
display(compare(glm_summary["std"], t_mcmc["sd"][1:] + [t_mcmc["total_sd"]], labels_glm).style.format(COMPARE_FMT))
display(compare(glm_summary["cov"] * 100, t_mcmc["cov_pct"][1:] + [t_mcmc["total_cov_pct"]], labels_glm).style.format(
    {"ours": "{:.1f}", "England": "{:.1f}", "diff %": "{:+.1f}%"}
))

Initializing NUTS using adapt_diag...


Multiprocess sampling (2 chains in 2 jobs)


NUTS: [alpha, Intercept, C(origin), C(dev)]


/Users/atroyer/Projects/bayesianchainladder/.claude/worktrees/stochastic-reserving-review-61d253/.venv/lib/python3.
12/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Sampling 2 chains for 1_000 tune and 1_000 draw iterations (2_000 + 2_000 draws total) took 1 seconds.


We recommend running at least 4 chains for robust computation of convergence diagnostics


,ours,England,diff %
2002,"123,978","94,497",+31.2%
2003,"500,305","473,377",+5.7%
2004,"685,928","713,963",-3.9%
2005,"1,081,981","994,634",+8.8%
2006,"1,562,686","1,427,398",+9.5%
2007,"2,328,026","2,185,862",+6.5%
2008,"3,823,204","3,939,219",-2.9%
2009,"4,356,397","4,327,368",+0.7%
2010,"4,806,496","4,730,275",+1.6%
Total,"19,269,000","18,886,591",+2.0%


,ours,England,diff %
2002,"73,279","110,589",-33.7%
2003,"197,932","216,833",-8.7%
2004,"218,553","264,946",-17.5%
2005,"308,653","308,600",+0.0%
2006,"404,891","378,706",+6.9%
2007,"586,547","502,610",+16.7%
2008,"1,039,124","801,520",+29.6%
2009,"1,293,178","1,074,894",+20.3%
2010,"1,973,945","2,040,253",-3.2%
Total,"3,083,935","3,024,716",+2.0%


,ours,England,diff %
2002,59.1,117.0,-49.5%
2003,39.6,45.8,-13.6%
2004,31.9,37.1,-14.1%
2005,28.5,31.0,-8.0%
2006,25.9,26.5,-2.2%
2007,25.2,23.0,+9.5%
2008,27.2,20.3,+33.9%
2009,29.7,24.8,+19.7%
2010,41.1,43.1,-4.7%
Total,16.0,16.0,+0.0%


**Commentary.** This is not the same model as England's: it shares the same
log-linear mean structure but uses a negative-binomial variance function
(`Var = mu + mu^2/k`) instead of the quasi-Poisson's over-dispersed variance
(`Var = phi * mu`), and weakly informative priors rather than flat ones. At
the total level the mean and SD are both within a few percent of England's
ODP MCMC figures. Per origin the gaps are larger — up to about 30% on
origin 2002 (the smallest, least mature reserve, most exposed to the
different variance function and prior) — and both directions appear: some
origins' SDs come out higher than England's, some lower, with no consistent
sign. This is the expected effect of comparing two different models sharing
only their mean structure, not a discrepancy to explain away.

## 7. MCMC Mack

In [7]:
mack_mcmc = BayesianMackChainLadder(**MCMC_KWARGS).fit(tri)
mack_negbin_mcmc = BayesianMackChainLadder(model="negbin", **MCMC_KWARGS).fit(tri)

devs = list(tri.development)
dev_ratio_labels = [f"{devs[i]}\u2192{devs[i + 1]}" for i in range(len(devs) - 1)]
factor_table = pd.DataFrame(
    {
        "chain ladder": mack_mcmc.factors_,
        "posterior mean (mack)": mack_mcmc.factor_draws_.mean(axis=0),
        "posterior sd (mack)": mack_mcmc.factor_draws_.std(axis=0),
        "posterior mean (negbin)": mack_negbin_mcmc.factor_draws_.mean(axis=0),
        "posterior sd (negbin)": mack_negbin_mcmc.factor_draws_.std(axis=0),
    },
    index=dev_ratio_labels,
)
display(factor_table.style.format("{:.4f}"))

t4_evw = ref_evw["table4_bootstrap_and_one_year_cdr"]
mack_summary = mack_mcmc.total_summary()
mack_negbin_summary = mack_negbin_mcmc.total_summary()
display(
    compare(
        [mack_summary.total_reserve_mean, mack_summary.total_reserve_stddev,
         mack_negbin_summary.total_reserve_mean, mack_negbin_summary.total_reserve_stddev],
        [t4_evw["total_avg_reserve"], t4_evw["total_bootstrap_sd"],
         t4_evw["total_avg_reserve"], t4_evw["total_bootstrap_sd"]],
        ["mack mean", "mack sd", "negbin mean", "negbin sd"],
    ).style.format(COMPARE_FMT)
)

Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (2 chains in 2 jobs)


NUTS: [coefs]


Sampling 2 chains for 1_000 tune and 1_000 draw iterations (2_000 + 2_000 draws total) took 0 seconds.


We recommend running at least 4 chains for robust computation of convergence diagnostics


...

NUTS: [coefs]


Sampling 2 chains for 1_000 tune and 1_000 draw iterations (2_000 + 2_000 draws total) took 1 seconds.


We recommend running at least 4 chains for robust computation of convergence diagnostics


,chain ladder,posterior mean (mack),posterior sd (mack),posterior mean (negbin),posterior sd (negbin)
12→24,3.4906,3.4741,0.2246,3.4664,0.2322
24→36,1.7473,1.7442,0.0592,1.7389,0.0605
36→48,1.4574,1.4552,0.0500,1.4490,0.0522
48→60,1.1739,1.1732,0.0281,1.1683,0.0280
60→72,1.1038,1.1033,0.0272,1.0944,0.0287
72→84,1.0863,1.0857,0.0220,1.0790,0.0238
84→96,1.0539,1.0539,0.0062,1.0531,0.0061
96→108,1.0766,1.0765,0.0115,1.0743,0.0122
108→120,1.0177,1.0174,0.0106,1.0131,0.0080


,ours,England,diff %
mack mean,"18,576,467","18,695,344",-0.6%
mack sd,"2,413,742","2,456,271",-1.7%
negbin mean,"17,552,377","18,695,344",-6.1%
negbin sd,"2,368,945","2,456,271",-3.6%


**Commentary.** England's EV 2006 notebook does not show Mack MCMC output,
so `BayesianMackChainLadder(model="mack")` (the exact Bayesian analogue of
his Mack Stan model from England & Verrall 2006, Section 6) is compared
instead to his EVW 2019 Mack *bootstrap* Table 4 totals — the paper states
these two should agree closely, which is what we check here, and it lands
within ordinary Monte Carlo error. The posterior mean factors track the
chain-ladder point estimates closely, as expected.

`BayesianMackChainLadder(model="negbin")` is the analogue of England's
Negative Binomial MCMC model, whose output his EV 2006 notebook also does
not show, so there is no published figure to compare its own posterior to
directly. Its total mean comes out several percent below the Mack bootstrap
benchmark; this is a genuine model difference, not a discrepancy to
reconcile. The per-cell likelihood variances of the two variants are
identical: `link_ratio_sigma` divides the residuals by `sqrt(f (f - 1))`
when estimating sigma and `build_link_ratio_model` multiplies it back, so
the `f (f - 1)` factor cancels. What differs is the parameterisation. The
negbin variant places its Normal prior on `log(log f)` and maps it back
through `exp(exp(.))`, which induces a different, asymmetric prior on the
factors and a different Jensen shift in the posterior mean than the Mack
variant's `exp(.)` map; the table above shows the resulting posterior mean
factors sitting slightly below the Mack variant's on every ratio, which
compounds into a lower total reserve.

## 8. Closing comparison

In [8]:
closing = pd.DataFrame(
    {
        "ours": [
            odp.total_sd,
            summary.loc["Total", "mean"],
            summary.loc["Total", "std"],
            posterior_mean[0],
            posterior_mean[-1],
            glm_summary.loc["Total", "mean"],
            glm_summary.loc["Total", "std"],
            mack_summary.total_reserve_mean,
            mack_summary.total_reserve_stddev,
        ],
        "England": [
            t_ml["total_sd"],
            t_boot["total_avg_reserve"],
            t_boot["total_sd"],
            mcmc_params["posterior_mean"][0],
            mcmc_params["posterior_mean"][-1],
            t_mcmc["total_avg_reserve"],
            t_mcmc["total_sd"],
            t4_evw["total_avg_reserve"],
            t4_evw["total_bootstrap_sd"],
        ],
    },
    index=[
        "ML total SD",
        "Bootstrap total mean",
        "Bootstrap total SD",
        "Quasi-Poisson posterior intercept",
        "Quasi-Poisson posterior DP10",
        "NB-GLM total mean",
        "NB-GLM total SD",
        "Bayesian Mack total mean",
        "Bayesian Mack total SD",
    ],
)
closing["diff %"] = 100 * (closing["ours"] - closing["England"]) / closing["England"]
display(closing.style.format({"ours": "{:,.2f}", "England": "{:,.2f}", "diff %": "{:+.1f}%"}))

,ours,England,diff %
ML total SD,"2,945,646.23","2,945,646.00",+0.0%
Bootstrap total mean,"18,860,425.82","18,866,138.00",-0.0%
Bootstrap total SD,"3,009,520.80","2,968,615.00",+1.4%
Quasi-Poisson posterior intercept,12.49,12.49,+0.0%
Quasi-Poisson posterior DP10,-1.80,-1.79,+0.3%
NB-GLM total mean,"19,269,000.09","18,886,591.00",+2.0%
NB-GLM total SD,"3,083,935.26","3,024,716.00",+2.0%
Bayesian Mack total mean,"18,576,466.90","18,695,344.00",-0.6%
Bayesian Mack total SD,"2,413,741.82","2,456,271.00",-1.7%


**Commentary, attributing each gap:**

* **ML total SD**: exact maximum-likelihood fit on both sides, so this
  matches England's to the display precision.
* **Bootstrap total mean/SD**: Gamma versus Normal pseudo-data in the
  resampling stage, plus ordinary Monte Carlo error at 10,000 simulations.
* **Quasi-Poisson posterior intercept/DP10**: MCMC sampling noise plus
  weakly informative `Normal(0, 10)` priors versus England's flat priors —
  largest on DP10, the least-identified coefficient.
* **NB-GLM total mean/SD**: a different variance function
  (`Var = mu + mu^2/k` versus ODP's `Var = phi * mu`) and weakly informative
  priors, on top of MCMC sampling noise — this is a genuinely different
  model, not a replica of England's ODP MCMC.
* **Bayesian Mack total mean/SD**: compared to EVW 2019's Mack bootstrap
  (the closest published benchmark, per the paper's own claim that the two
  agree), so on top of MCMC/MC noise there is also a small
  bootstrap-vs-MCMC estimation-method gap.

No gap in this table falls outside these causes.